In [1]:
!pip install -q numbers-parser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.0/326.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires prot

In [2]:
from numbers_parser import Document
import pandas as pd
import numpy as np

file_path = (
    "/kaggle/input/datasets/isabellacsiu/"
    "breakthroughtech-training-data-1updated/"
    "BreakThroughTech_training_data_1.numbers"
)

document = Document(file_path)
table = document.sheets[0].tables[0]

data = table.rows(values_only=True)

df = pd.DataFrame(
    data[1:],
    columns=data[0]
)

df.head()

,brand,sub_product_name,country,pillar,media_type,profession,specialty,campaign_start_date,unique_reach,unique_engaged_hcps,...,post_avg_NRx,pre_NRx_avg,npi_specialty,region,NRx_lift,NRx_lift_pct,cpe,total_spend,actual_spend_per_engagement,spend_per_NRx_lift
0,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,8.0,undefined,2024-07-31 10:48:34.000000 UTC,9270.0,1070.0,...,72.8343,47.4714,Neurology,Southeast,146.77,15.5000,39.0,134355308.0,30198.99,915440.73
1,ABBVIE,BRAND ALERT,usa,search,EDITORIAL_VIDEOS,8.0,undefined,2024-08-19 17:16:44.000000 UTC,1959.0,380.0,...,54.1279,62.6833,Cardiology,Southwest,112.05,12.2398,39.0,10667850.0,9055.90,95202.91
2,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,psychologist,urology,2023-11-16 15:54:10.000000 UTC,2860.0,403.0,...,42.1019,66.2213,Cardiology,Mid-Atlantic,78.27,10.8516,39.0,59103000.0,34705.23,755161.42
3,ABBVIE,BRAND ALERT,zaf,search,SPONSORED_VIDEOS,8.0,undefined,2024-11-22 10:27:31.000000 UTC,7390.0,910.0,...,49.5424,55.8293,Neurology,Northeast,121.87,11.9201,39.0,124487739.0,112455.05,1021479.17
4,ABBVIE,EMAIL ALERT,unknown,unknown,None,unknown,None,2023-11-16 17:38:18.000000 UTC,77570.0,26264.0,...,46.8907,39.3346,Endocrinology,Southwest,221.51,35.1116,3.0,892409925.0,7691.40,4028833.69


In [3]:
campaign_features = [
    "unique_reach",
    "unique_engaged_hcps",
    "total_engagements",
    "total_budget_spent",
    "engagement_rate",
    "engagement_frequency",
]

df[campaign_features].describe().T

,count,mean,std,min,25%,50%,75%,max
unique_reach,417.0,27259.611511,50388.227523,514.000,1748.0000,7258.0000,31371.0000,291110.000
unique_engaged_hcps,417.0,7675.347722,17297.149112,49.000,399.0000,1616.0000,6076.0000,141813.000
total_engagements,417.0,22028.467626,48786.838688,135.000,1093.0000,4390.0000,18420.0000,395199.000
total_budget_spent,417.0,393328.503861,931105.555291,0.000,17862.0000,77883.0000,375180.0000,9364485.000
engagement_rate,417.0,0.261123,0.125172,0.056,0.1541,0.2397,0.3502,0.549
engagement_frequency,417.0,2.921367,0.928341,1.200,2.1900,2.9600,3.6900,4.500


In [4]:
df[campaign_features].isna().sum()

unique_reach            0
unique_engaged_hcps     0
total_engagements       0
total_budget_spent      0
engagement_rate         0
engagement_frequency    0
dtype: int64

In [5]:
df["calculated_engagement_rate"] = (
    df["unique_engaged_hcps"]
    / df["unique_reach"]
)

df["calculated_engagement_frequency"] = (
    df["total_engagements"]
    / df["unique_engaged_hcps"]
)

df[
    [
        "engagement_rate",
        "calculated_engagement_rate",
        "engagement_frequency",
        "calculated_engagement_frequency",
    ]
].head()

,engagement_rate,calculated_engagement_rate,engagement_frequency,calculated_engagement_frequency
0,0.1154,0.115426,4.16,4.157944
1,0.1940,0.193977,3.10,3.100000
2,0.1409,0.140909,4.23,4.225806
3,0.1231,0.123139,1.22,1.216484
4,0.3386,0.338585,4.42,4.417720


In [6]:
# Nearly all missing CPE records imply a CPE of $39.
validation_cpe = df["cpe"].fillna(39)

expected_budget = (
    df["total_engagements"]
    * validation_cpe
)

budget_error = ~np.isclose(
    df["total_budget_spent"],
    expected_budget,
    atol=0.01,
    rtol=0,
)

budget_issues = df.loc[
    budget_error,
    [
        "brand",
        "sub_product_name",
        "total_engagements",
        "cpe",
        "total_budget_spent",
    ]
].copy()

budget_issues.insert(
    0,
    "excel_row",
    budget_issues.index + 2
)

budget_issues["expected_budget"] = expected_budget[budget_error]

budget_issues

,excel_row,brand,sub_product_name,total_engagements,cpe,total_budget_spent,expected_budget
109,111,GILEAD,BRAND ALERT,209.0,39.0,39.94,8151.0
114,116,GILEAD,EMAIL ALERT,757.0,3.0,24.26,2271.0
119,121,GILEAD,TARGET LIST MEDIA,372.0,NaN,7.06,14508.0
143,145,JANSSEN BIOTECH,BRAND ALERT,184.0,39.0,0.77,7176.0
146,148,JANSSEN BIOTECH,EMAIL ALERT,643.0,3.0,0.00,1929.0
173,175,LILLY,CONDITION MESSAGE,595.0,11.0,0.00,6545.0
227,229,MERCK,BRAND ALERT,1398.0,39.0,299.59,54522.0
236,238,MERCK,BRAND ALERT,332.0,39.0,98.49,12948.0


In [7]:
# Preserve the original dataframe.
eda_df = df.copy()

eda_df["budget_quality_flag"] = np.where(
    budget_error,
    "Review",
    "Valid",
)

# Keep the original budget and create a separate cleaned version.
eda_df["clean_total_budget_spent"] = (
    eda_df["total_budget_spent"]
)

eda_df.loc[
    budget_error,
    "clean_total_budget_spent"
] = np.nan

In [8]:
eda_df["log_unique_reach"] = np.log1p(
    eda_df["unique_reach"]
)

eda_df["log_total_budget_spent"] = np.log1p(
    eda_df["clean_total_budget_spent"]
)

eda_df[
    [
        "unique_reach",
        "log_unique_reach",
        "total_budget_spent",
        "clean_total_budget_spent",
        "log_total_budget_spent",
        "engagement_rate",
        "engagement_frequency",
        "budget_quality_flag",
    ]
].head()

,unique_reach,log_unique_reach,total_budget_spent,clean_total_budget_spent,log_total_budget_spent,engagement_rate,engagement_frequency,budget_quality_flag
0,9270.0,9.134647,173511.0,173511.0,12.064002,0.1154,4.16,Valid
1,1959.0,7.580700,45942.0,45942.0,10.735157,0.1940,3.10,Valid
2,2860.0,7.958926,66417.0,66417.0,11.103723,0.1409,4.23,Valid
3,7390.0,8.908018,43173.0,43173.0,10.672994,0.1231,1.22,Valid
4,77570.0,11.258949,348081.0,348081.0,12.760193,0.3386,4.42,Valid


In [9]:
# Rename columns in the cleaned dataframe only.
eda_df = eda_df.rename(columns={
    "total_budget_spent": "budget_spent_to_date",
    "clean_total_budget_spent": "clean_budget_spent_to_date",
    "log_total_budget_spent": "log_budget_spent_to_date",
    "total_spend": "total_campaign_budget",
})

eda_df[[
    "budget_spent_to_date",
    "clean_budget_spent_to_date",
    "total_campaign_budget",
]].head()

,budget_spent_to_date,clean_budget_spent_to_date,total_campaign_budget
0,173511.0,173511.0,134355308.0
1,45942.0,45942.0,10667850.0
2,66417.0,66417.0,59103000.0
3,43173.0,43173.0,124487739.0
4,348081.0,348081.0,892409925.0


In [10]:
# Convert the start date to datetime. Invalid or missing dates become NaT.
eda_df["campaign_start_date"] = pd.to_datetime(
    eda_df["campaign_start_date"],
    errors="coerce",
    utc=True,
)

eda_df["campaign_start_date_missing"] = (
    eda_df["campaign_start_date"].isna()
)

budget_missing = eda_df["total_campaign_budget"].isna()

can_estimate_budget = (
    budget_missing
    & eda_df["actual_spend_per_engagement"].notna()
    & eda_df["total_engagements"].notna()
)

eda_df["total_campaign_budget_source"] = "Original"

eda_df.loc[
    can_estimate_budget,
    "total_campaign_budget",
] = (
    eda_df.loc[
        can_estimate_budget,
        "actual_spend_per_engagement",
    ]
    * eda_df.loc[can_estimate_budget, "total_engagements"]
).round(2)

eda_df.loc[
    can_estimate_budget,
    "total_campaign_budget_source",
] = "Estimated from spend per engagement"

missing_record_review = eda_df.loc[
    budget_missing | eda_df["campaign_start_date_missing"],
    [
        "brand",
        "sub_product_name",
        "campaign_start_date",
        "budget_spent_to_date",
        "total_campaign_budget",
        "total_campaign_budget_source",
    ],
].copy()

missing_record_review.insert(
    0,
    "excel_row",
    missing_record_review.index + 2,
)

missing_record_review

,excel_row,brand,sub_product_name,campaign_start_date,budget_spent_to_date,total_campaign_budget,total_campaign_budget_source
170,172,LILLY,BRANDSPOT,NaT,994228.0,1.737848e+08,Estimated from spend per engagement
183,185,LILLY,TARGET LIST MEDIA,NaT,229515.0,1.737848e+08,Estimated from spend per engagement
282,284,MERCK,INFOSITE,NaT,128115.0,1.737848e+08,Estimated from spend per engagement


In [11]:
brand_map = {
    "ADC_THERAPEUTICS_ZYNLONTA": "ADC THERAPEUTICS",
    "AMGEN_HORIZON_KRYSTEXXA": "AMGEN",
    "ASTELLAS_XTANDI": "ASTELLAS",
    "ASTRAZENECA_ENHERTU": "ASTRAZENECA",
    "BMS": "BRISTOL-MYERS SQUIBB",
    "BOEHRINGER-INGELHEIM": "BOEHRINGER INGELHEIM",
    "DAY_ONE_BIO": "DAY ONE BIOPHARMACEUTICALS",
    "HORIZON": "HORIZON THERAPEUTICS",
    "SERONO": "EMD SERONO",
    "SERONO_TEPMETKO": "EMD SERONO",
    "J&J": "JOHNSON & JOHNSON",
    "JANSSEN": "JOHNSON & JOHNSON",
    "JANSSEN BIOTECH": "JOHNSON & JOHNSON",
    "JANSSEN CNS": "JOHNSON & JOHNSON",
    "JANSSEN PHARMACEUTICALS": "JOHNSON & JOHNSON",
    "JANSSEN_DARZALEX": "JOHNSON & JOHNSON",
    "JANSSEN_DARZALEX_MDALERT": "JOHNSON & JOHNSON",
    "JANSSEN_ERLEADA": "JOHNSON & JOHNSON",
    "JANSSEN_INVEGA": "JOHNSON & JOHNSON",
    "JANSSEN_RYBREVANT": "JOHNSON & JOHNSON",
    "JANSSEN_SPRAVATO": "JOHNSON & JOHNSON",
    "JANSSEN_TALVEY": "JOHNSON & JOHNSON",
    "JANSSEN_TECVAYLI": "JOHNSON & JOHNSON",
    "JANSSEN_TREMFYAPSA": "JOHNSON & JOHNSON",
    "JANSSEN_TREMFYAPSO": "JOHNSON & JOHNSON",
    "JAZZ_ZEPZELCA_BA": "JAZZ PHARMACEUTICALS",
    "JAZZ_ZEPZELCA_EA": "JAZZ PHARMACEUTICALS",
    "LILLY_MOUNJARO_MDALERT": "LILLY",
    "LILLY_MOUNJARO_MEDSCAPE": "LILLY",
    "NOVONORDISK_OZEMPIC": "NOVO NORDISK",
    "NOVONORDISK_WEGOVY": "NOVO NORDISK",
    "PFIZER_ELREXFIO": "PFIZER",
    "PFIZER_PAXLOVID": "PFIZER",
    "RIGEL": "RIGEL PHARMACEUTICALS",
    "SEAGEN_ADCETRIS_MDALERT": "SEAGEN",
    "SEAGEN_ADCETRIS_MEDSCAPE": "SEAGEN",
    "SEAGEN_TIVDAK": "SEAGEN",
    "SEAGEN_TIVDAK_MDALERT": "SEAGEN",
    "SUMITOMO_GEMTESA": "SUMITOMO PHARMA",
    "TAKEDA_ZS": "TAKEDA",
    "VIIV": "VIIV HEALTHCARE",
}

eda_df["brand_clean"] = (
    eda_df["brand"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace(brand_map)
)

print("Original brand names:", eda_df["brand"].nunique())
print("Clean company names:", eda_df["brand_clean"].nunique())

brand_mapping_review = (
    eda_df[["brand", "brand_clean"]]
    .drop_duplicates()
    .sort_values(["brand_clean", "brand"])
)

brand_mapping_review

Original brand names: 73
Clean company names: 34


,brand,brand_clean
0,ABBVIE,ABBVIE
8,ADC THERAPEUTICS,ADC THERAPEUTICS
11,ADC_THERAPEUTICS_ZYNLONTA,ADC THERAPEUTICS
14,AMGEN,AMGEN
16,AMGEN_HORIZON_KRYSTEXXA,AMGEN
...,...,...
388,TAKEDA,TAKEDA
396,TAKEDA_ZS,TAKEDA
405,TEVA,TEVA
406,VIIV,VIIV HEALTHCARE


In [12]:
pd.Series({
    "rows": len(eda_df),
    "raw_brand_count": eda_df["brand"].nunique(),
    "clean_brand_count": eda_df["brand_clean"].nunique(),
    "estimated_campaign_budgets": (
        eda_df["total_campaign_budget_source"]
        == "Estimated from spend per engagement"
    ).sum(),
    "missing_campaign_budgets": (
        eda_df["total_campaign_budget"].isna().sum()
    ),
    "missing_start_dates": (
        eda_df["campaign_start_date_missing"].sum()
    ),
})

rows                          417
raw_brand_count                73
clean_brand_count              34
estimated_campaign_budgets      3
missing_campaign_budgets        0
missing_start_dates             3
dtype: int64

In [13]:
# Save all the cleaning and normalization results
output_path = "/kaggle/working/jin_cleaned_data.csv"

eda_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(eda_df))
print("Columns:", len(eda_df.columns))

Saved: /kaggle/working/jin_cleaned_data.csv
Rows: 417
Columns: 35


In [14]:
feature_df = eda_df.copy()

# Log versions of baseline prescriptions and engagement counts
for column in [
    "pre_NRx_avg",
    "unique_engaged_hcps",
    "total_engagements"
]:
    feature_df[f"log_{column}"] = np.log1p(
        feature_df[column].where(feature_df[column] >= 0)
    )

# Use directly recorded budgets, excluding the three reconstructed values
original_budget = feature_df["total_campaign_budget"].where(
    feature_df["total_campaign_budget_source"] == "Original"
)

feature_df["log_total_campaign_budget"] = np.log1p(
    original_budget.where(original_budget >= 0)
)

In [15]:
spent = feature_df["clean_budget_spent_to_date"]
engaged = feature_df["unique_engaged_hcps"]

feature_df["cost_per_engaged_hcp"] = (
    spent / engaged.where(engaged > 0)
)

In [16]:
new_features = [
    "log_total_campaign_budget",
    "log_pre_NRx_avg",
    "log_unique_engaged_hcps",
    "log_total_engagements",
    "cost_per_engaged_hcp"
]

display(feature_df[new_features].head())
display(feature_df[new_features].isna().sum())

,log_total_campaign_budget,log_pre_NRx_avg,log_unique_engaged_hcps,log_total_engagements,cost_per_engaged_hcp
0,18.715998,3.880974,6.976348,8.400659,162.159813
1,16.182745,4.153922,5.942799,7.072422,120.900000
2,17.894792,4.207990,6.001415,7.440734,164.806452
3,18.639718,4.040052,6.814543,7.010312,47.442857
4,20.609436,3.697210,10.175993,11.661587,13.253160


log_total_campaign_budget    3
log_pre_NRx_avg              0
log_unique_engaged_hcps      0
log_total_engagements        0
cost_per_engaged_hcp         8
dtype: int64

In [17]:
feature_df.to_csv(
    "/kaggle/working/jin_cleaned_features.csv",
    index=False
)

print("Saved jin_cleaned_features.csv")

Saved jin_cleaned_features.csv


In [18]:
feature_df["cpe"] = feature_df["cpe"].fillna(
    0
)

print("Missing CPE:", feature_df["cpe"].isna().sum())

Missing CPE: 0


In [19]:

feature_df["log_pre_NRx"] = np.log1p(
    feature_df["pre_NRx"].where(feature_df["pre_NRx"] >= 0)
)

- `planned_budget_per_baseline_NRx`: planned budget divided by prescriptions before the campaign. It uses only original budget values and skips zero baselines.
- `start_quarter`: the quarter when the campaign started (1–4), or `Unknown` if the date is missing.

In [20]:
model_df = feature_df.copy()


recorded_budget = model_df["total_campaign_budget"].where(
    model_df["total_campaign_budget_source"] == "Original"
)

baseline = model_df["pre_NRx"]

model_df["planned_budget_per_baseline_NRx"] = (
    recorded_budget / baseline.where(baseline > 0)
)

model_df["start_quarter"] = (
    model_df["campaign_start_date"]
    .dt.quarter
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

- It separates categorical columns from numeric columns and copies them into `X`.
- Missing category values become `Unknown`.
- One-hot encoding turns each category into 0/1 columns. The last two lines show the new table’s size and first few rows.

In [21]:
categorical_cols = [
    "brand_clean",
    "sub_product_name",
    "pillar",
    "media_type",
    "profession",
    "npi_specialty",
    "country",
    "start_quarter",
]

model_df[categorical_cols] = (
    model_df[categorical_cols].astype("string").fillna("Unknown")
)

model_df = pd.get_dummies(
    model_df,
    columns=categorical_cols,
    dtype=int
)

display(model_df.head())

,brand,specialty,campaign_start_date,unique_reach,unique_engaged_hcps,total_engagements,budget_spent_to_date,engagement_rate,engagement_frequency,pre_NRx,...,country_vir,country_vnm,country_zaf,country_zmb,country_zwe,start_quarter_1,start_quarter_2,start_quarter_3,start_quarter_4,start_quarter_Unknown
0,ABBVIE,undefined,2024-07-31 10:48:34+00:00,9270.0,1070.0,4449.0,173511.0,0.1154,4.16,946.88,...,0,0,0,0,0,0,0,1,0,0
1,ABBVIE,undefined,2024-08-19 17:16:44+00:00,1959.0,380.0,1178.0,45942.0,0.1940,3.10,915.49,...,0,0,0,0,0,0,0,1,0,0
2,ABBVIE,urology,2023-11-16 15:54:10+00:00,2860.0,403.0,1703.0,66417.0,0.1409,4.23,721.23,...,0,0,0,0,0,0,0,0,1,0
3,ABBVIE,undefined,2024-11-22 10:27:31+00:00,7390.0,910.0,1107.0,43173.0,0.1231,1.22,1022.39,...,0,0,1,0,0,0,0,0,1,0
4,ABBVIE,None,2023-11-16 17:38:18+00:00,77570.0,26264.0,116027.0,348081.0,0.3386,4.42,630.86,...,0,0,0,0,0,0,0,0,1,0


In [22]:
# initial discussed features from 9/29 meeting after setting a baseline

model_df["budget_per_reach"] = model_df["budget_spent_to_date"] / model_df["unique_reach"]
model_df["budget_per_baseline_NRx"] = model_df["budget_spent_to_date"] / model_df["pre_NRx_avg"]
model_df["log_totalSpend"] = np.log1p(model_df["total_campaign_budget"])
model_df["log_reach"] = np.log1p(model_df["unique_reach"])

In [23]:
# new discussed feature implementations from 10/4 meeting

# reach compared to market size 
# it shows the # of HCPs reached for every avg baseline prescription
model_df["reach_vs_baseline"] = model_df["unique_reach"] / model_df["pre_NRx_avg"]

# engaged HCPs compared to market size
model_df["engaged_vs_baseline"] = model_df["unique_engaged_hcps"] / model_df["pre_NRx_avg"]

# money spent so far for each extra prescription gained
model_df["budget_per_nrx_lift"] = model_df["budget_spent_to_date"] / model_df["NRx_lift"]

# average prescriptions gained for each unit of engagement rate
model_df["avg_nrx_gain_per_engagement_rate"] = (model_df["post_avg_NRx"] - model_df["pre_NRx_avg"]) / model_df["calculated_engagement_rate"]


In [24]:
model_df

,brand,specialty,campaign_start_date,unique_reach,unique_engaged_hcps,total_engagements,budget_spent_to_date,engagement_rate,engagement_frequency,pre_NRx,...,start_quarter_4,start_quarter_Unknown,budget_per_reach,budget_per_baseline_NRx,log_totalSpend,log_reach,reach_vs_baseline,engaged_vs_baseline,budget_per_nrx_lift,avg_nrx_gain_per_engagement_rate
0,ABBVIE,undefined,2024-07-31 10:48:34+00:00,9270.0,1070.0,4449.0,173511.0,0.1154,4.16,946.88,...,0,0,18.717476,3655.063891,18.715998,9.134647,195.275471,22.539887,1182.196634,219.732788
1,ABBVIE,undefined,2024-08-19 17:16:44+00:00,1959.0,380.0,1178.0,45942.0,0.1940,3.10,915.49,...,0,0,23.451761,732.922485,16.182745,7.580700,31.252343,6.062221,410.013387,-44.105338
2,ABBVIE,urology,2023-11-16 15:54:10+00:00,2860.0,403.0,1703.0,66417.0,0.1409,4.23,721.23,...,1,0,23.222727,1002.955242,17.894792,7.958926,43.188521,6.085655,848.562668,-171.169935
3,ABBVIE,undefined,2024-11-22 10:27:31+00:00,7390.0,910.0,1107.0,43173.0,0.1231,1.22,1022.39,...,1,0,5.842084,773.303624,18.639718,8.908018,132.367771,16.299685,354.254534,-51.055155
4,ABBVIE,None,2023-11-16 17:38:18+00:00,77570.0,26264.0,116027.0,348081.0,0.3386,4.42,630.86,...,1,0,4.487315,8849.231974,20.609436,11.258949,1972.055137,667.707311,1571.400840,22.316733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,VIIV,None,2023-11-16 17:38:18+00:00,38093.0,19149.0,27085.0,81255.0,0.5027,1.41,366.89,...,1,0,2.133069,4502.210796,20.865302,10.547812,2110.672769,1061.015747,583.225668,22.780406
413,VIIV,undefined,2023-05-02 13:53:22+00:00,2569.0,682.0,1722.0,5166.0,0.2655,2.52,381.32,...,0,0,2.010899,348.587700,17.866154,7.851661,173.349168,46.019514,52.108130,144.584224
414,VIIV,undefined,2024-07-31 10:48:34+00:00,53210.0,13593.0,53893.0,161679.0,0.2555,3.96,587.67,...,0,0,3.038508,7937.308230,20.408140,10.882020,2612.238887,667.321240,1047.482993,67.483887
415,VIIV,undefined,2024-03-04 14:27:38+00:00,1185.0,544.0,1066.0,3198.0,0.4591,1.96,602.33,...,0,0,2.698734,132.392757,14.803873,7.078342,49.057354,22.520844,30.396350,25.492095


In [25]:
model_df.to_csv(
    "/kaggle/working/combined_dataset.csv",
    index=False
)

print("Saved combined_dataset.csv")

Saved combined_dataset.csv
